# semaine 5

extract data

In [ ]:
import pandas as pd

df = pd.read_csv('microdados_ed_basica_2024_SP.csv', sep=';', encoding='latin1', low_memory=False)

print(df.info())
print(df.head())

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

#separer binaires et non binaires
binary_cols = [c for c in df.columns if c.startswith('IN_')]
count_cols = [c for c in df.columns if c.startswith('QT_')]
other_cols = [c for c in df.columns if not c.startswith('IN_') and not c.startswith('QT_')]

#on étudie bianires et counts
subset_cols = binary_cols[:20] + count_cols[:10]
data_subset = df[subset_cols].copy()

#imputer
imputer = SimpleImputer(strategy='median')
data_imputed = pd.DataFrame(imputer.fit_transform(data_subset), columns=subset_cols)

#on sépare
binary_data = data_imputed[[c for c in subset_cols if c.startswith('IN_')]]
non_binary_data = data_imputed[[c for c in subset_cols if c.startswith('QT_')]]

print(f"nb binaires {len(binary_data.columns)}")
print(f"nb non binaires {len(non_binary_data.columns)}")

#on garde les autres
binary_data.to_csv('binary_data.csv', index=False)
non_binary_data.to_csv('non_binary_data.csv', index=False)

#on tente un cluster k means
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_imputed)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(data_scaled)
data_imputed['Cluster'] = clusters

print("\nClustering:")
print(data_imputed['Cluster'].value_counts())

In [ ]:

from sklearn.linear_model import LinearRegression
import networkx as nx
import matplotlib.pyplot as plt

#on sélectione pour le test
df = pd.read_csv('microdados_ed_basica_2024_SP.csv', sep=';', encoding='latin1', low_memory=False)

selected_cols = [
    'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
    'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS', 'QT_DOC_BAS']

imputer = SimpleImputer(strategy='median')
X_clean = pd.DataFrame(imputer.fit_transform(df[selected_cols]), columns=selected_cols)

binary_data = X_clean[[c for c in selected_cols if c.startswith('IN_')]]
non_binary_data = X_clean[[c for c in selected_cols if not c.startswith('IN_')]]

binary_data.to_csv('binary_data_final.csv', index=False)
non_binary_data.to_csv('non_binary_data_final.csv', index=False)

#clustering (maybe optional?)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
X_clean['Cluster'] = clusters
X_clean.to_csv('clustered_results.csv', index=False)

#rex
def rex_algorithm(data_df):
    X = data_df.values.copy()
    names = data_df.columns.tolist()
    n = len(names)
    remaining = list(range(n))
    order = []
    
    #iterate
    while len(remaining) > 0:
        scores = []
        for i in remaining:
            total_dep = 0
            xi = X[:, i].reshape(-1, 1)
            for j in remaining:
                if i == j: continue
                xj = X[:, j].reshape(-1, 1)
                lr = LinearRegression().fit(xi,xj)
                residual = xj - lr.predict(xi)
                total_dep += abs(np.corrcoef(xi.flatten(), residual.flatten())[0, 1])
            scores.append((total_dep, i))
        
        scores.sort()
        best_root = scores[0][1]
        order.append(best_root)
        remaining.remove(best_root)
        if len(remaining) > 0:
            xi_root = X[:, best_root].reshape(-1, 1)
            for j in remaining:
                xj = X[:, j].reshape(-1, 1)
                lr = LinearRegression().fit(xi_root, xj)
                X[:, j] = (xj - lr.predict(xi_root)).flatten()
                
    return [names[i] for i in order]

causal_order = rex_algorithm(X_clean.drop(columns=['Cluster']))
print("Causal Order Found:", causal_order)

#Construct DAG X = BX + e
B = pd.DataFrame(0.0, index=selected_cols, columns=selected_cols)
X_std_df = pd.DataFrame(X_scaled, columns=selected_cols)

for i, target in enumerate(causal_order):
    if i == 0: continue
    parents = causal_order[:i]
    
    y = X_std_df[target]
    X_p = X_std_df[parents]
    
    reg = LinearRegression().fit(X_p, y)
    for p_idx, p_name in enumerate(parents):
        B.loc[target, p_name] = reg.coef_[p_idx]


B_pruned = B.copy()
B_pruned[np.abs(B_pruned) < 0.1] =0


G = nx.DiGraph()
for node in selected_cols:
    G.add_node(node)

for row in B_pruned.index:
    for col in B_pruned.columns:
        if B_pruned.loc[row, col] != 0:
            G.add_edge(col, row, weight=round(B_pruned.loc[row, col], 2))

plt.figure(figsize=(14, 8))
pos = nx.spring_layout(G, k=2)
nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=4000, font_size=9, arrowsize=25)
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.title("DAG")
plt.tight_layout()
plt.savefig('rex_causal_dag.png')

In [ ]:

lien= 'microdados_ed_basica_2024_SP.csv'
selected_cols = [
        'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS', 'QT_DOC_BAS']


def graph(lien, cols):
    df = pd.read_csv(lien, sep=';', encoding='latin1', low_memory=False)

    imputer = SimpleImputer(strategy='median')
    X_clean = pd.DataFrame(imputer.fit_transform(df[cols]), columns=cols)

    binary_data = X_clean[[c for c in cols if c.startswith('IN_')]]
    non_binary_data = X_clean[[c for c in cols if not c.startswith('IN_')]]

    binary_data.to_csv('binary_data_final.csv', index=False)
    non_binary_data.to_csv('non_binary_data_final.csv', index=False)

    #clustering (maybe optional?)
    #scaler = StandardScaler()
    #X_scaled = scaler.fit_transform(X_clean)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    X_clean['Cluster'] = clusters
    X_clean.to_csv('clustered_results.csv', index=False)

    #rex
    def rex_algorithm(data_df):
        X = data_df.values.copy()
        names = data_df.columns.tolist()
        n = len(names)
        remaining = list(range(n))
        order = []
        
        #iterate
        while len(remaining) > 0:
            scores = []
            for i in remaining:
                total_dep= 0
                xi = X[:, i].reshape(-1, 1)
                for j in remaining:
                    if i == j: continue
                    xj = X[:, j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi,xj)
                    residual = xj - lr.predict(xi)
                    total_dep += abs(np.corrcoef(xi.flatten(), residual.flatten())[0,1])
                scores.append((total_dep, i))
            
            scores.sort()
            best_root = scores[0][1]
            order.append(best_root)
            remaining.remove(best_root)
            if len(remaining) >0:
                xi_root = X[:, best_root].reshape(-1, 1)
                for j in remaining:
                    xj = X[:,j].reshape(-1, 1)
                    lr = LinearRegression().fit(xi_root, xj)
                    X[:, j] = (xj - lr.predict(xi_root)).flatten()
                    
        return [names[i] for i in order]

    causal_order = rex_algorithm(X_clean.drop(columns=['Cluster']))
    print("causal ordre", causal_order)

    #Construct DAG X = BX + e
    B = pd.DataFrame(0.0, index=cols, columns=cols)
    X_std_df = pd.DataFrame(X_scaled, columns=cols)

    for i, target in enumerate(causal_order):
        if i == 0: continue
        parents = causal_order[:i]
        
        y = X_std_df[target]
        X_p = X_std_df[parents]
        
        reg= LinearRegression().fit(X_p, y)
        for p_idx, p_name in enumerate(parents):
            B.loc[target, p_name] = reg.coef_[p_idx]


    B_pruned= B.copy()
    B_pruned[np.abs(B_pruned) < 0.1] =0


    G= nx.DiGraph()
    for node in selected_cols:
        G.add_node(node)

    for row in B_pruned.index:
        for col in B_pruned.columns:
            if B_pruned.loc[row, col]!= 0:
                G.add_edge(col, row, weight=round(B_pruned.loc[row, col], 2))

    plt.figure(figsize=(14, 8))
    pos = nx.spring_layout(G, k=2)
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=4000, font_size=9, arrowsize=25)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
    plt.title("DAG")
    plt.tight_layout()
    plt.savefig('rex_causal_dag.png')



lien= 'microdados_ed_basica_2024_SP.csv'
selected_cols = [
        'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_INTERNET',
        'IN_LABORATORIO_INFORMATICA', 'QT_SALAS_UTILIZADAS', 'QT_MAT_BAS', 'QT_DOC_BAS']

graph(lien, selected_cols)

Fracture numérique

In [ ]:
selected_cols2=["IN_INTERNET_APRENDIZAGEM", "QT_COMP_PORTATIL_ALUNO", "IN_BANDA_LARGA", "QT_MAT_PROF_TEC"]
lien= 'microdados_ed_basica_2024_SP.csv'
graph(lien, selected_cols2)

In [ ]:
selected_cols2=["IN_ACESSIBILIDADE_RAMPAS", "IN_ACESSIBILIDADE_ELEVADOR", "IN_ACESSIBILIDADE_SINAL_TATIL", "QT_MAT_ESP"]
lien= 'microdados_ed_basica_2024_SP.csv'
graph(lien, selected_cols2)

In [ ]:
selected_cols2=["IN_LABORATORIO_CIENCIAS", "IN_LABORATORIO_INFORMATICA", "IN_MATERIAL_PED_CIENTIFICO", "QT_MAT_MED"]
lien= 'microdados_ed_basica_2024_SP.csv'
graph(lien, selected_cols2)

In [ ]:
selected_cols2=["IN_LABORATORIO_CIENCIAS", "IN_LABORATORIO_INFORMATICA", "IN_MATERIAL_PED_CIENTIFICO", "QT_MAT_MED"] #qtt élèves
lien= 'microdados_ed_basica_2024_SP.csv'
graph(lien, selected_cols2)